<a href="https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

### Research question

Can a machine-learning model prioritize search-performance opportunities as effectively as the existing Week-4 rule-based baseline?

### Decision supported

The output is intended to help analysts decide which search-performance observations should be reviewed first.

The system is a decision-support tool. It does not automatically decide which SEO changes should be made.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Research question: ML prioritization vs Week-4 baseline")
print("Decision: prioritize observations for human review")


Research question: ML prioritization vs Week-4 baseline
Decision: prioritize observations for human review


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

The analysis uses the FlyRank ML Internship search-performance dataset.

The working sample contains 50,000 observations. The data contains aggregate search,
traffic, engagement, and AI-referral signals.

Client names, domains, private queries, and personally identifying information are
not used.

Client and content identifiers are also excluded from the predictive feature set.

The working observations are used to construct an 80/20 chronological train/test split.

In [4]:
!pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully")

Paste your Hugging Face READ token: ··········
Connected successfully


In [5]:
test = con.execute(
    f"SELECT * FROM {TABLES['fact_daily_sample']} LIMIT 5"
).df()

print(test.shape)
display(test.head())

(5, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 3. Methodology

### Target

The target is the Week-4 rule-based action category generated from search-performance
signals:

- HIGH_IMPRESSIONS_LOW_CTR
- RANKING_IMPROVEMENT
- GROWTH_OPPORTUNITY
- STABLE_PERFORMER

### Features

The Random Forest uses aggregate performance signals including impressions, clicks,
search position, page views, sessions, users, and traffic-source sessions.

Client identifiers, content identifiers, private queries, and target-derived fields
are excluded.

### Baseline

The Week-4 rule-based prioritization system is used as the baseline.

### Validation

The data is sorted chronologically and divided into 40,000 training observations and
10,000 test observations. The same test observations are used for both the baseline
and Random Forest.

### Leakage checks

Fields that directly encode target information, future information, private queries,
or identifiers are excluded from predictive features.

The evaluation measures whether the model reproduces the predefined action categories.
It does not measure whether an SEO recommendation causes better search performance.

In [8]:
# Load the public-safe sample dataset

import pandas as pd
import numpy as np

df = con.execute(
    f"""
    SELECT *
    FROM {TABLES['fact_daily_sample']}
    LIMIT 50000
    """
).df()

df["report_date"] = pd.to_datetime(df["report_date"])

print("Dataset shape:", df.shape)
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())
print("Columns:", len(df.columns))

display(df.head())

Dataset shape: (50000, 31)
Date range: 2026-06-01 00:00:00 to 2026-06-01 00:00:00
Columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [9]:
# Feature engineering

df["ctr"] = (
    df["gsc_clicks"] /
    (df["gsc_impressions"] + 1)
)

high_imp = df["gsc_impressions"].quantile(0.75)
median_ctr = df["ctr"].median()
median_imp = df["gsc_impressions"].median()

df["reason_code"] = np.select(
    [
        (df["gsc_impressions"] > high_imp) &
        (df["ctr"] < median_ctr),

        df["gsc_sum_position"] > 10,

        df["gsc_impressions"] > median_imp
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CTR",
        "RANKING_IMPROVEMENT",
        "GROWTH_OPPORTUNITY"
    ],
    default="STABLE_PERFORMER"
)

print(df["reason_code"].value_counts())

reason_code
STABLE_PERFORMER       39877
RANKING_IMPROVEMENT     8065
GROWTH_OPPORTUNITY      2058
Name: count, dtype: int64


In [10]:
split_index = int(len(df) * 0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]

In [11]:
# Time-ordered train/test split

df = df.sort_values("report_date").reset_index(drop=True)

split_index = int(len(df) * 0.8)

train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

print("Training observations:", len(train))
print("Test observations:", len(test))

print("Training period:",
      train["report_date"].min(),
      "to",
      train["report_date"].max())

print("Test period:",
      test["report_date"].min(),
      "to",
      test["report_date"].max())

Training observations: 40000
Test observations: 10000
Training period: 2026-06-01 00:00:00 to 2026-06-01 00:00:00
Test period: 2026-06-01 00:00:00 to 2026-06-01 00:00:00


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results (vs baseline)

The Week-4 baseline and Random Forest were evaluated on exactly the same 10,000 test
observations using weighted F1.

The Random Forest matched the baseline at 1.00 F1.

Therefore, the experiment did not demonstrate a measurable improvement from Random
Forest under this evaluation setup.

In [13]:
# Define model features

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai"
]

X_train = train[features]
y_train = train["reason_code"]

X_test = test[features]
y_test = test["reason_code"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (40000, 13)
X_test: (10000, 13)
y_train: (40000,)
y_test: (10000,)


In [15]:
from sklearn.metrics import f1_score

# The baseline already generated the labels.
baseline_pred = y_test.copy()

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    average="weighted"
)

print("Baseline F1:", baseline_f1)

Baseline F1: 1.0


In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

model_f1 = f1_score(
    y_test,
    model_pred,
    average="weighted"
)

print("Random Forest F1:", model_f1)

Random Forest F1: 1.0


In [17]:
results = pd.DataFrame({
    "Approach": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Training observations": [
        len(X_train),
        len(X_train)
    ],
    "Test observations": [
        len(X_test),
        len(X_test)
    ],
    "F1 Score": [
        baseline_f1,
        model_f1
    ]
})

display(results)

,Approach,Training observations,Test observations,F1 Score
0,Week-4 Baseline,40000,10000,1.0
1,Random Forest,40000,10000,1.0


## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

This experiment has several important limitations.

- The target is generated from predefined rules, so the Random Forest is largely
  learning to reproduce the baseline.
- The 1.00 F1 result applies only to the reported evaluation split.
- The experiment does not establish causality.
- It does not predict Google's ranking algorithm.
- It does not guarantee increased clicks, traffic, rankings, or conversions.
- The model has not demonstrated an advantage over the simpler baseline.
- Additional unseen time periods and independently observed outcomes would provide a
  stronger test of whether machine learning adds practical value.

The results should therefore be treated as decision-support evidence rather than proof
of future SEO performance.

In [19]:
# Final evaluation summary

errors = int((model_pred != y_test).sum())

print("Random Forest errors:", errors)
print("Baseline F1:", baseline_f1)
print("Random Forest F1:", model_f1)
print("F1 improvement:", model_f1 - baseline_f1)

Random Forest errors: 0
Baseline F1: 1.0
Random Forest F1: 1.0
F1 improvement: 0.0


In [20]:
results = pd.DataFrame({
    "Approach": ["Week-4 Baseline", "Random Forest"],
    "Training observations": [len(X_train), len(X_train)],
    "Test observations": [len(X_test), len(X_test)],
    "F1 Score": [baseline_f1, model_f1]
})

display(results)

,Approach,Training observations,Test observations,F1 Score
0,Week-4 Baseline,40000,10000,1.0
1,Random Forest,40000,10000,1.0


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Ranked recommendations

The ranked output is intended to support human review rather than automatically decide which SEO action should be taken.

The highest-priority observations are reviewed first using the action associated with their reason code.

Recommendations are:

1. Review high-priority opportunities first.
2. For `HIGH_IMPRESSIONS_LOW_CTR`, review the title, meta description, and search snippet.
3. For `RANKING_IMPROVEMENT`, review the content and opportunities to improve search position.
4. For `GROWTH_OPPORTUNITY`, investigate whether the content can be expanded or aligned with related search demand.
5. Validate recommendations against business context, seasonality, and tracking quality before taking action.

The Random Forest matched the Week-4 baseline on the evaluated F1 metric, so the baseline remains a strong and interpretable reference rather than being replaced solely because a machine-learning model was used.



In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Generate ranked recommendations from the test observations

recommendations = test.copy()

recommendations["priority"] = model_pred

def action_from_reason(reason):
    if reason == "HIGH_IMPRESSIONS_LOW_CTR":
        return "Improve title/meta description and search snippet"
    elif reason == "RANKING_IMPROVEMENT":
        return "Optimize content to improve ranking"
    elif reason == "GROWTH_OPPORTUNITY":
        return "Expand content and target related queries"
    else:
        return "Monitor performance"

recommendations["action"] = recommendations["priority"].apply(
    action_from_reason
)

recommendations["confidence_note"] = (
    "Review manually before taking action"
)

ranked_recommendations = recommendations[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position",
        "ctr",
        "priority",
        "action",
        "confidence_note"
    ]
].copy()

ranked_recommendations = ranked_recommendations.head(20)

display(ranked_recommendations)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,ctr,priority,action,confidence_note
40000,2026-06-01,client_625b6439094e23e4,content_09b82b3006e90020,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40001,2026-06-01,client_625b6439094e23e4,content_1c225a6ecfd8daea,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40002,2026-06-01,client_625b6439094e23e4,content_04428ecba307cea1,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40003,2026-06-01,client_625b6439094e23e4,content_73080c4ef67a6ac8,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40004,2026-06-01,client_625b6439094e23e4,content_57071ad6f783e473,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40005,2026-06-01,client_625b6439094e23e4,content_93e070fe27e6183f,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40006,2026-06-01,client_625b6439094e23e4,content_1d8410347fb636ed,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40007,2026-06-01,client_625b6439094e23e4,content_312b0faebd95bee8,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40008,2026-06-01,client_625b6439094e23e4,content_4a3f5ff4c10c34d4,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action
40009,2026-06-01,client_625b6439094e23e4,content_191d1041becb6e8d,0,0,0,0.0,STABLE_PERFORMER,Monitor performance,Review manually before taking action


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Artifacts the paper embeds

The capstone produces a small set of reproducible artifacts supporting the reported findings:

- Model-versus-baseline evaluation table
- Reason-code distribution
- Ranked top-20 recommendations

These artifacts are descriptive outputs from the experiment and are not treated as evidence of causal improvement.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create output directory

import os

os.makedirs("work/outputs", exist_ok=True)

# 1. Save evaluation results

results.to_csv(
    "work/outputs/capstone_results.csv",
    index=False
)

# 2. Reason-code distribution

reason_counts = (
    df["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

reason_counts.to_csv(
    "work/outputs/reason_code_distribution.csv",
    index=False
)

# 3. Save top-20 recommendations

ranked_recommendations.to_csv(
    "work/outputs/capstone_top20_recommendations.csv",
    index=False
)

# 4. Feature importance

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

feature_importance.to_csv(
    "work/outputs/feature_importance.csv",
    index=False
)

print("Artifacts created:")
print(" - capstone_results.csv")
print(" - reason_code_distribution.csv")
print(" - capstone_top20_recommendations.csv")
print(" - feature_importance.csv")

Artifacts created:
 - capstone_results.csv
 - reason_code_distribution.csv
 - capstone_top20_recommendations.csv
 - feature_importance.csv


In [23]:
# Display the most important model features

display(
    feature_importance.head(10)
)

,feature,importance
2,gsc_sum_position,0.425533
0,gsc_impressions,0.313510
3,gsc_avg_position,0.213362
4,ga4_pageviews,0.017022
6,ga4_users,0.011688
5,ga4_sessions,0.009902
7,sessions_organic,0.005119
1,gsc_clicks,0.002218
8,sessions_direct,0.000963
9,sessions_referral,0.000428


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.